##  Tool Calling · Agent · Structured Output · MCP

LLM 단독 응답의 한계를 확인하고, Python 도구·에이전트·출력 검증·원격 MCP 도구를 연결합니다. 

### 핵심 개념
- **LLM은 Python 함수를 스스로 실행하지 않습니다.** 모델이 도구 이름과 인자를 제안하면 *애플리케이션 런타임*이 이를 실행합니다.
- **`@tool`**: 함수의 타입 힌트와 docstring으로 도구 메타데이터와 인자 스키마를 만듭니다.
- **`bind_tools()`**: 모델에게 "이런 도구들이 있다"고 알려주어 호출(Function Calling)을 유도합니다.
- **`create_agent()`**: 모델 판단 → 도구 실행 → ToolMessage 전달 → 모델 재호출을 관리합니다. \
  &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**내부적으로 LangGraph 기반 실행 그래프를 만듭니다.**
- **`response_format=`**: `structured_response`에 검증된 구조화 결과를 반환합니다.
- **MCP**: 외부 MCP 서버가 제공하는 도구를 표준 프로토콜로 가져와 에이전트에 연결합니다.

> **한눈에 보기**
> - LLM은 도구를 **결정**만 하고 실행은 코드가 한다. `@tool`(타입 힌트 + docstring → 스키마)
> - `create_agent(model, tools, system_prompt)` → 도구 선택·실행·재호출 루프 자동화 (LangGraph `CompiledStateGraph`)
> - `response_format=Pydantic` → `result["structured_response"]` (형식만 보장, 사실은 따로 확인)
> - OpenAI 내장 `web_search`(서버에서 실행) vs `@tool`(내 프로그램에서 실행), 원격 **MCP** 서버 도구 연결
> - 정리 문서: [도구호출과MCP.md](도구호출과MCP.md) · 슬라이드 이미지는 자리만 표시, 웹 검색 결과는 앞부분만 남김, 📝 정리 셀은 직접 정리

In [1]:
# 환경 및 패키지 버전 확인 (온라인 호출 없음)
from importlib.metadata import version, PackageNotFoundError
for name in ["langchain", "langchain-core", "langchain-openai", "langchain-mcp-adapters", "pydantic"]:
    try:
        print(f"{name}: {version(name)}")
    except PackageNotFoundError:
        print(f"{name}: 설치되지 않음")


langchain: 1.4.6
langchain-core: 1.6.12
langchain-openai: 1.7.2
langchain-mcp-adapters: 0.3.2
pydantic: 2.14.1


### 1. 도구가 없는 LLM vs 도구가 있는 LLM

- LLM이 현재 시간이나 날씨를 자체 지식만으로 답하면 틀릴 수 있습니다. 
- 실제 시간/날씨는 신뢰할 수 있는 외부 데이터에서 가져와야 합니다. 
- 아래 첫 실습은 **도구 미사용 상황의 한계**를 확인합니다.

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
response = llm.invoke("지금 몇 시야? 그리고 오늘 날짜도 알려줘.")
print(response.content)
print("응답 객체:", type(response).__name__)

죄송하지만, 현재 시간을 확인하거나 날짜를 알려드릴 수는 없습니다. 하지만 오늘 날짜나 시간을 확인하려면 기기나 시계를 참고하시면 됩니다. 다른 질문이 있으시면 도와드리겠습니다!
응답 객체: AIMessage


**📝 정리** — 도구가 없으니 모델이 현재 시각을 모른다고 답했다. 지금 바뀌는 값(시간, 날씨)은 외부 데이터에서 가져와야 한다.

### 2. `@tool`: Python 함수에서 도구 스키마 생성

- 타입 힌트는 입력 인자 타입을, 함수의 docstring은 도구가 하는 일을 모델에 알려줍니다. 
- 함수 본문은 **모델이 아니라 Python 실행 환경에서** 동작합니다.

In [3]:
from datetime import datetime
from zoneinfo import ZoneInfo
from langchain_core.tools import tool

@tool
def get_current_time() -> str:
    """현재 한국 표준시(Asia/Seoul)의 날짜와 시각을 반환합니다."""
    return datetime.now(ZoneInfo("Asia/Seoul")).strftime("%Y-%m-%d %H:%M:%S %Z")

print("이름:", get_current_time.name)
print("설명:", get_current_time.description)
print("인자 스키마:", get_current_time.args)
print("Python 도구 직접 실행:", get_current_time.invoke({}))

이름: get_current_time
설명: 현재 한국 표준시(Asia/Seoul)의 날짜와 시각을 반환합니다.
인자 스키마: {}
Python 도구 직접 실행: 2026-10-19 14:15:32 KST


**📝 정리** — 함수 이름 → 도구 이름, docstring → 설명, 타입 힌트 → 인자 스키마(인자가 없어 `{}`). `invoke({})`로 도구만 따로 실행해 볼 수 있다.

### 3. `create_agent()`: 자동 도구 실행 루프

- `create_agent(model, tools, system_prompt)`는 도구 선택과 반복 호출을 관리합니다. 
- 내부 메시지에는 `AIMessage`의 `tool_calls`, 실행 결과를 기록하는 `ToolMessage` 등이 포함됩니다.

> `agent.invoke({"messages": [...]})`의 결과는 **상태 딕셔너리**입니다. 최종 응답 텍스트는 `result["messages"][-1].content`입니다.

In [4]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# 🚀 도구 호출 루프 전체를 create_agent가 대신 처리합니다.
agent = create_agent(
    model=llm,
    tools=[get_current_time],
    system_prompt="당신은 친절한 어시스턴트입니다. 정보를 답변할 때 필요하다면 제공된 도구를 적극적으로 사용하세요.",
)

user_question = "지금 몇 시야? 그리고 오늘 날짜도 알려줘."
print(f"사용자 질문: {user_question}\n")
print("--- AI Agent 실행 ---")

result = agent.invoke({"messages": [{"role": "user", "content": user_question}]})

# 실행 과정에서 오간 모든 메시지(도구 호출/결과 포함)를 확인할 수 있습니다.

for m in result["messages"]:
    print(f"[{type(m).__name__}] {getattr(m, 'content', '')}")
    if getattr(m, "tool_calls", None):
        print(f"   -> 도구 호출: {m.tool_calls}")

print("\n--- AI Agent 종료 ---")
print(f"\n최종 AI 답변: {result['messages'][-1].content}")

사용자 질문: 지금 몇 시야? 그리고 오늘 날짜도 알려줘.

--- AI Agent 실행 ---
[HumanMessage] 지금 몇 시야? 그리고 오늘 날짜도 알려줘.
[AIMessage] 
   -> 도구 호출: [{'name': 'get_current_time', 'args': {}, 'id': 'call_du700W1GekoDnTNIacZnnL1S', 'type': 'tool_call'}]
[ToolMessage] 2026-10-19 14:19:11 KST
[AIMessage] 현재 시각은 2026년 10월 19일 오후 2시 19분입니다. 도움이 더 필요하신가요?

--- AI Agent 종료 ---

최종 AI 답변: 현재 시각은 2026년 10월 19일 오후 2시 19분입니다. 도움이 더 필요하신가요?


**📝 정리** — 메시지 흐름: `HumanMessage` → 내용이 빈 `AIMessage`(`tool_calls`만 있음) → `ToolMessage`(도구 실행 결과) → 최종 `AIMessage`. 이 루프를 `create_agent`가 대신 돌렸다.

In [5]:
type(agent)

langgraph.graph.state.CompiledStateGraph

**📝 정리** — `create_agent`의 결과가 LangGraph 그래프(`CompiledStateGraph`)다. LangChain v1의 에이전트는 내부가 LangGraph라는 것을 확인했다.

#### Agent의 메시지 흐름

사용자 → ChatOpenAI(도구 선택) → `AIMessage.tool_calls` → Python `@tool` 실행 → `ToolMessage` → ChatOpenAI(최종 답변).

도구가 필요하지 않다고 판단하면 도구 호출 없이 바로 종료할 수 있습니다.

### **create_agent()**
LangChain v1 버전부터 도입된 `create_agent()`는 기존에 LangGraph에서 사용하던 `create_react_agent`를 대체하고 더 유연한 미들웨어(Middleware) 시스템을 결합하여 에이전트 구축을 대폭 단순화한 핵심 팩토리 함수입니다.

복잡한 상태 그래프(StateGraph)를 직접 수동으로 짜지 않고도, 모델과 도구만 넘겨주면 알아서 루프(Loop)를 돌며 도구를 호출하고 결과를 취합하는 ReAct 에이전트 그래프를 생성해 줍니다.

**create_agent()의 핵심 특징**   

- 통합된 API: langchain.agents 패키지를 통해 직관적으로 에이전트를 정의할 수 있습니다.   
- 유연한 미들웨어(Middleware) 구조: 프롬프트 제어, 로깅, 재시도, 보안 검증 등을 미들웨어 단위로 끼워 넣을 수 있어 확장성이 뛰어납니다.   
- 강력한 제어 기능: 체크포인터(메모리)와 Human-in-the-Loop(사람의 승인 개입) 설정을 손쉽게 통합할 수 있습니다.  

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

### 4. 구조화 출력: `response_format`과 `structured_response`

LangChain 1.x `create_agent(..., response_format=MyModel)`은 Pydantic 스키마를 검증하고 결과를 **`result["structured_response"]`** 에 보관합니다. 모델/제공자에 따라 구조화 출력 전략(`ProviderStrategy`, `ToolStrategy`)이 달라질 수 있습니다. <https://docs.langchain.com/oss/python/langchain/structured-output>

In [6]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

class DateTimeResponse(BaseModel):
    current_year: int = Field(description="연도")
    current_month: int = Field(description="월 (1~12)", ge=1, le=12)
    current_date: int = Field(description="일 (1~31)", ge=1, le=31)
    current_hour: int = Field(description="시 (0~23)", ge=0, le=23)
    current_minute: int = Field(description="분 (0~59)", ge=0, le=59)
    current_second: int = Field(description="초 (0~59)", ge=0, le=59)

agent = create_agent(
    model=ChatOpenAI(model="gpt-4o-mini", temperature=0),
    tools=[get_current_time],
    system_prompt="날짜/시간 답변 전에는 반드시 get_current_time 도구를 사용하세요. 한국 시간 기준으로 답하세요.",
    response_format=DateTimeResponse,
)
result = agent.invoke({"messages": [{"role": "user", "content": "한국 현재 날짜와 시간을 알려줘."}]})

# 중요: 모델의 마지막 응답 content를 JSON으로 가정하지 않습니다.
structured = result["structured_response"]
print("자료형:", type(structured).__name__)
print(structured.model_dump())
print("연도:", structured.current_year, "시간:", structured.current_hour)


자료형: DateTimeResponse
{'current_year': 2026, 'current_month': 10, 'current_date': 19, 'current_hour': 15, 'current_minute': 7, 'current_second': 37}
연도: 2026 시간: 15


**📝 정리** — `structured_response`에 Pydantic 검증을 통과한 `DateTimeResponse` 객체가 들어 있다. 마지막 메시지 content를 JSON으로 직접 파싱할 필요가 없다.

In [18]:
import json
import requests
from langchain_core.tools import tool

@tool
def get_current_weather(location: str) -> str:
    """지정한 도시의 현재 날씨, 기온(°C), 습도(%)를 외부 날씨 API에서 조회합니다.
    Args:
        location: 영문 도시명. 예: Seoul, Tokyo, London, Suwon
    """
    try:
        response = requests.get(
            f"https://wttr.in/{requests.utils.quote(location, safe='')}?format=j1",
            timeout=10,
            headers={"User-Agent": "LangChain-classroom/1.0"},
        )
        response.raise_for_status()
        current = response.json()["current_condition"][0]
        return json.dumps({
            "location": location,
            "temperature_celsius": current["temp_C"],
            "condition": current["weatherDesc"][0]["value"],
            "humidity_percent": current["humidity"],
        }, ensure_ascii=False)
    except (requests.RequestException, ValueError, KeyError, IndexError) as exc:
        return json.dumps({"error": f"날씨 API 조회 실패: {type(exc).__name__}"}, ensure_ascii=False)

print(get_current_weather.args)
# 외부 서버의 장애, 요청 제한, 응답 구조 변경 등으로 실패할 수 있습니다.


{'location': {'title': 'Location', 'type': 'string'}}


**📝 정리** — 외부 API(wttr.in)가 실패하면 예외를 던지지 않고 `{"error": ...}` JSON을 돌려준다. docstring에 "영문 도시명. 예: Seoul"을 적어서 모델이 한글 지명을 영문으로 바꿔 넘기게 했다.

In [19]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

def main():
    llm = ChatOpenAI(model="gpt-4o-mini")

    agent = create_agent(
        model=llm,
        tools=[get_current_weather],
        system_prompt="당신은 기상 캐스터 어시스턴트입니다. 제공된 도구를 사용해 실시간 정보를 얻어 사용자에게 친절하게 안내하세요.",
    )

    user_question = "지금 서울 날씨 어때? 그리고 런던 날씨도 같이 확인해줘."
    print(f"💬 사용자 질문: {user_question}\n")
    print("🤖 Agent 실행 시작...")

    # 💡 서울/런던 두 도시 모두를 위해 도구를 두 번 호출하는 것까지 create_agent가 알아서 처리합니다.
    result = agent.invoke({"messages": [{"role": "user", "content": user_question}]})

    print("\n✨ 최종 AI 답변:")
    print(result["messages"][-1].content)

if __name__ == "__main__":
    main()


💬 사용자 질문: 지금 서울 날씨 어때? 그리고 런던 날씨도 같이 확인해줘.

🤖 Agent 실행 시작...

✨ 최종 AI 답변:
현재 서울의 날씨는 흐림(Overcast)이며, 기온은 19°C, 습도는 35%입니다.

런던의 경우, 맑음(Clear) 상태로 기온은 10°C, 습도는 83%입니다.

더 궁금한 점이 있으시면 언제든지 말씀해 주세요!


**📝 정리** — 최종 답만 출력해서 도구를 몇 번 불렀는지는 보이지 않지만, 서울(19°C 흐림)과 런던(10°C 맑음) 값을 모두 도구에서 가져왔다. 메시지 흐름을 보려면 7번 셀처럼 `result["messages"]`를 출력한다.

### 도구 호출을 단계별로 살펴보기
1. 앱이 모델에 질문과 도구 스키마를 전달합니다.
2. 모델은 도구가 필요하면 **도구 이름과 인자**를 반환합니다 (`AIMessage.tool_calls`).
3. 앱/Agent 런타임이 Python 함수를 실행하고 결과를 `ToolMessage`로 기록합니다.
4. 모델이 도구 결과를 반영해 최종 답변을 작성합니다. 도구 호출이 반복될 수도 있습니다.

**중요**: `bind_tools()`는 도구 정보 바인딩까지 담당하고, `create_agent()`는 실행 루프까지 담당합니다.

**LLM의 역할: "이 함수를 이런 값으로 실행해라" 하고 결정(Decision)하는 역할.**

**애플리케이션(코드)의 역할: LLM의 결정을 보고 파이썬 함수를 실제 실행(Execution)하는 역할.**

**Schema 작성 원칙**

- 좋은 Tool Schema는 “모델이 추측하지 않아도 되게” 만드는 게 핵심입니다.

JSON Schema로 직접 Tool 정의를 작성하는 것도 가능하지만, 이 실습에서는 **`@tool` + 명확한 타입 힌트/docstring**을 권장합니다. `strict` 여부나 지원 방식은 모델 제공자와 API 버전에 따라 다릅니다.

좋은 도구 설명에는 **언제 사용해야 하는지, 인자에 허용되는 값, 예상 반환 데이터, 실패 조건**을 포함하세요.

In [20]:
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# 🚀 도구 호출 루프 전체를 create_agent가 대신 처리합니다.
agent = create_agent(
    model=llm,
    tools=[get_current_weather, get_current_time],
    system_prompt="당신은 친절한 어시스턴트입니다. 정보를 답변할 때 필요하다면 제공된 도구를 적극적으로 사용하세요.",
)

user_question = "경기도 수원 날씨 알려줘 그리고 현재 시간도 알려줘"
print(f"사용자 질문: {user_question}\n")
print("--- AI Agent 실행 ---")

result = agent.invoke({"messages": [{"role": "user", "content": user_question}]})

# 실행 과정에서 오간 모든 메시지(도구 호출/결과 포함)를 확인할 수 있습니다.

for m in result["messages"]:
    print(f"[{type(m).__name__}] {getattr(m, 'content', '')}")
    if getattr(m, "tool_calls", None):
        print(f"   -> 도구 호출: {m.tool_calls}")

print("\n--- AI Agent 종료 ---")
print(f"\n최종 AI 답변: {result['messages'][-1].content}")

사용자 질문: 경기도 수원 날씨 알려줘 그리고 현재 시간도 알려줘

--- AI Agent 실행 ---
[HumanMessage] 경기도 수원 날씨 알려줘 그리고 현재 시간도 알려줘
[AIMessage] 
   -> 도구 호출: [{'name': 'get_current_weather', 'args': {'location': 'Suwon'}, 'id': 'call_AmZLopDJCv93DxbSjexnFLMm', 'type': 'tool_call'}, {'name': 'get_current_time', 'args': {}, 'id': 'call_bRe8bDH5tenwLUafzzmlNnaZ', 'type': 'tool_call'}]
[ToolMessage] {"location": "Suwon", "temperature_celsius": "19", "condition": "Overcast ", "humidity_percent": "33"}
[ToolMessage] 2026-10-19 15:18:53 KST
[AIMessage] 현재 경기도 수원의 날씨는 다음과 같습니다:
- 기온: 19°C
- 날씨: 흐림
- 습도: 33%

현재 한국 표준시는 2026년 10월 19일 15시 18분 53초입니다.

--- AI Agent 종료 ---

최종 AI 답변: 현재 경기도 수원의 날씨는 다음과 같습니다:
- 기온: 19°C
- 날씨: 흐림
- 습도: 33%

현재 한국 표준시는 2026년 10월 19일 15시 18분 53초입니다.


**📝 정리** — 첫 `AIMessage`가 날씨 도구와 시간 도구를 **동시에** 요청했다. "경기도 수원"은 docstring 예시대로 `Suwon`으로 바뀌어 넘어갔다.

### 다중 도구 호출의 주의사항

모델이 한 번에 여러 도구 호출을 요청할 수 있으나 실제 호출 횟수나 순서는 보장되지 않습니다. 날씨 서비스 장애 시 오류 메시지를 그대로 날씨 정보인 것처럼 말하지 않도록 프롬프트를 지정하세요.

## 6. OpenAI 내장 도구와 Python 도구는 다릅니다

`@tool`은 Python 애플리케이션이 실행하는 **로컬 도구**입니다. OpenAI의 `web_search` 등 제공자 내장 도구는 OpenAI 측에서 수행되며 도구별 **지원 모델·요금·사용 조건**이 다릅니다. LangChain OpenAI 통합에서 Responses API를 명시적으로 사용할 수 있습니다.

아래 웹 검색 예제는 계정/모델에서 지원되는 경우에만 실행하세요. **`gpt-4o-mini`가 특정 OpenAI 빌트인 도구를 반드시 지원한다고 가정하지 않습니다.**

In [21]:
from langchain_openai import ChatOpenAI

# 빌트인 web_search는 계정·모델 지원 여부에 따라 오류가 발생할 수 있습니다.
# 사용 가능 여부를 확인한 뒤 아래 셀을 실행하세요.
web_model = ChatOpenAI(model="gpt-4o-mini", use_responses_api=True)
web_model = web_model.bind_tools([{"type": "web_search_preview"}])

try:
    response = web_model.invoke("오늘 AI 분야의 중요한 소식을 출처와 함께 요약해줘.")
    print("응답 내용:", response.content)
    print("응답 블록 자료형:", type(response.content).__name__)
    # response.content[1]['text'] 같은 고정 인덱스는 사용하지 않습니다.
except Exception as exc:
    print("이 모델/계정에서 내장 웹 검색을 사용할 수 없거나 요청에 실패했습니다:", exc)


응답 내용: [{'id': 'ws_0518f4e162ec2519006ad5b6b045cc87d0be937c9cdd196555', 'action': {'type': 'search', 'queries': ['today AI news'], 'query': 'today AI news'}, 'status': 'completed', 'type': 'web_search_call'}, {'type': 'text', 'text': '최근 인공지능(AI) 분야에서 주목할 만한 소식들을 정리해 드리겠습니다.\n\n**1. 금융권의 AI 도입 가속화**\n\nKB, 신한, 하나, 우리, 농협 등 주요 금융그룹들이 AI 전환(AX)을 핵심 전략으로 삼아 여신 심사, 자산 관리, 고객 상담 등 전 업무에 AI를 본격 도입하고 있습니다. 특히 우리은행은 약 884억 원 규모의 AI 에이전트 구축 사업을 추진하며, 신한금융은 현장형 AI 인력 양성에 집중하고 있습니다. ([newspim.com](https://www.newspim.com/news/view/20260306000918?utm_source=openai))\n\n**2. 퀄컴과 아마존의 차세대 AI 칩 개발 협력**\n\n퀄컴은 아마존웹서비스(AWS)와 최대 600억 달러(약 80조 원) 규모의 차세대 AI 칩 개발 협약을 체결했습니다. 이를 통해 스마트폰을 넘어 데이터센터로 사업 영역을 확장하고자 합니다. ([yna.co.kr](https://www.yna.co.kr/view/AKR20260909004151091?utm_source=openai))\n\n... (이하 생략 — 웹 검색 결과 요약이라 앞부분만 남김)
응답 블록 자료형: list


**📝 정리** — Responses API를 쓰면 `content`가 블록 **리스트**다(`web_search_call` + 출처가 달린 `text`). "오늘 소식"을 물었지만 출처 날짜는 3월·6월·9월 기사였고 AI와 관계없는 채용 소식도 섞였다. 검색 도구를 붙여도 날짜 조건은 직접 확인해야 한다.

## 7. MCP (Model Context Protocol)

`@tool`은 현재 프로그램 안의 함수를, **MCP**는 외부 서버가 공개하는 도구를 연결합니다. `MultiServerMCPClient`가 서버에서 도구 목록을 가져오고 `create_agent`가 이를 사용할 수 있습니다. MCP 서버가 설치되어 있고 신뢰 가능한지, 접근 권한과 데이터 전송 범위가 적절한지 먼저 확인해야 합니다.

### 원격 MCP 사용 전 확인할 사항
- 전송 방식: `streamable_http` 또는 `stdio` 등 서버 방식에 맞춰 설정합니다.
- 원격 주소는 **실제 운영되고 신뢰할 수 있는 서버의 URL**이 필요합니다.
- `get_tools()`는 서버 연결에 실패하면 예외가 발생합니다.
- 반환된 도구 이름/설명을 확인하고 개인정보나 API 키를 도구 인자로 보내지 마세요.

공식 참고: <https://docs.langchain.com/oss/python/langchain/mcp>

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

**Open-Meteo MCP Server란?**
- Open-Meteo MCP Server는 ChatGPT, Claude, Cursor 같은 AI 클라이언트가 날씨 데이터를 조회할 수 있도록 연결해 주는 MCP 서버입니다. 
- 현재 날씨뿐 아니라 시간별 예보, 과거 기상 데이터, 대기질, 해양 날씨 등도 조회할 수 있습니다. 
- 가장 큰 장점은 비상업적 용도라면 날씨 API 키 없이 무료로 사용할 수 있고, Streamable HTTP를 지원하는 공개 서버도 있다는 점입니다.

In [22]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
import os

# 외부 공개 MCP 서비스 주소는 운영 종료·변경될 수 있습니다.
# 직접 구축한 신뢰할 수 있는 MCP 서버 URL을 환경 변수로 지정하세요.
MCP_URL = "https://open-meteo.caseyjhand.com/mcp"

async def run_mcp_demo():
    if not MCP_URL:
        print("실습을 건너뜁니다: WEATHER_MCP_URL 환경 변수를 MCP 서버 주소로 설정하세요.")
        return
    client = MultiServerMCPClient({
        "weather": {"url": MCP_URL, "transport": "streamable_http"}
    })
    try:
        tools = await client.get_tools()
        print("MCP 제공 도구:", [t.name for t in tools])
        if not tools:
            print("서버에 사용 가능한 도구가 없습니다.")
            return
        agent = create_agent(
            model=ChatOpenAI(model="gpt-4o-mini", temperature=0),
            tools=tools,
            system_prompt="제공된 도구로 확인한 사실과 확인하지 못한 사실을 구분해 답하세요.",
        )
        result = await agent.ainvoke({
            "messages": [{"role": "user", "content": "서울 날씨 알려줘."}]
        })
        print(result["messages"][-1].content)
    except Exception as exc:
        print("MCP 도구 연결/실행 오류:", type(exc).__name__, str(exc))

await run_mcp_demo()  # Jupyter 최상위 await 지원


MCP 제공 도구: ['openmeteo_search_locations', 'openmeteo_get_elevation', 'openmeteo_get_forecast', 'openmeteo_get_historical', 'openmeteo_get_marine', 'openmeteo_get_air_quality', 'openmeteo_get_ensemble', 'openmeteo_get_flood', 'openmeteo_get_climate', 'openmeteo_dataframe_query', 'openmeteo_dataframe_describe']
현재 서울의 날씨는 다음과 같습니다:

- **온도:** 17.4°C
- **강수량:** 0 mm (비 없음)
- **풍속:** 4.8 km/h
- **날씨 코드:** 맑음

날씨는 맑고 기온은 17.4도입니다.


**📝 정리** — 공개 Open-Meteo MCP 서버에서 도구 11개를 받아 와 서울 날씨를 17.4°C 맑음이라고 답했다. 같은 시각 wttr.in은 19°C 흐림이었다 → 데이터 출처가 다르면 값도 다르다.

## 8. 실무에서 꼭 구분해야 할 API 비교

| API | 역할 | 결과 |
|---|---|---|
| `@tool` | Python 함수를 도구로 선언 | 스키마를 가진 Tool 객체 |
| `llm.bind_tools([...])` | 모델에 사용 가능 도구 등록 | **도구 호출 제안** (`AIMessage.tool_calls`), 실행은 직접 구현해야 함 |
| `create_agent(...)` | 도구 선택·실행·재호출 자동화 | 메시지 목록을 포함하는 상태 |
| `response_format=MyModel` | 최종 결과 스키마 지정 | `result["structured_response"]` |
| `MultiServerMCPClient.get_tools()` | 외부 MCP 도구를 가져옴 | `BaseTool` 호환 도구 리스트 |
| `agent.ainvoke(...)` | 비동기 실행 | 상태 객체 |

**주의**: 구조화 출력은 응답 형식을 검증하지만 **사실의 정확성을 보장하지는 않습니다**. 시간/날씨 정보는 사용한 도구의 결과와 일치하는지 따로 확인해야 합니다.

## 공식 참고 문서
- [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents)
- [Structured output](https://docs.langchain.com/oss/python/langchain/structured-output)
- [ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai)
- [MCP](https://docs.langchain.com/oss/python/langchain/mcp)

> 🖼️ *(강의 슬라이드 이미지 — 강사님 자료라 저장소에는 넣지 않음)*

## 🔁 복습 질문

<details>
<summary>1. <code>bind_tools()</code>와 <code>create_agent()</code>의 차이는?</summary>

`bind_tools()`는 모델에 도구를 알려 줘서 호출 제안(`tool_calls`)만 받는다. `create_agent()`는 실행과 재호출 루프까지 돌린다.
</details>

<details>
<summary>2. 도구 실행 결과는 어떤 메시지로 모델에 돌아가나?</summary>

`ToolMessage`.
</details>

<details>
<summary>3. <code>response_format</code>을 지정한 에이전트의 검증된 결과는 어디 있나?</summary>

`result["structured_response"]`.
</details>

<details>
<summary>4. MCP 도구와 <code>@tool</code> 도구의 차이는?</summary>

`@tool`은 내 프로그램 안의 함수, MCP 도구는 외부 서버가 표준 프로토콜로 공개한 도구다. 에이전트에는 둘 다 같은 도구 리스트로 들어간다.
</details>
